# Sanjivani Rural Education Society's
## Sanjivani College of Engineering, Kopargaon-423603
### Department of Information Technology
**Class:** B. Tech. Semester: VII  
**Subject:** Advanced Data Science Laboratory (PEIT412B)  
**Practical No.: 06**  
**CO3:** Apply MLOps and deployment techniques for scalable AI systems. (BTL-Apply)  

---
### **Title:** Deployment of Scalable AI Systems using Docker, Managing AI Services, and Integrating Deployment Pipelines.
---


## 1. Learning Outcomes & Objectives
- Understand containerization concepts and the role of **Docker** in deploying AI systems.
- Write a `Dockerfile` to containerize Machine Learning and Deep Learning applications.
- Build, tag, and execute Docker images/containers for AI services.
- Manage multi-service AI applications using **Docker Compose** (`docker-compose.yml`).
- Implement Continuous Integration and Continuous Deployment (**CI/CD**) pipelines using **GitHub Actions**.

## 2. Exemplars / Case Studies Implemented
1. **AI Attendance Monitoring System**: Single-container face-recognition-based attendance system converting camera features into model inference and timestamp logs.
2. **Smart Retail Analytics Platform**: Multi-service AI application (Retail Footfall API + PostgreSQL DB + Streamlit Dashboard) orchestrated together using **Docker Compose**.


## 3. Case Study 1: AI Attendance Monitoring System Containerization

### A. Dockerfile (`ai_attendance_system/Dockerfile`)
```dockerfile
FROM python:3.10-slim

WORKDIR /app

RUN apt-get update && apt-get install -y --no-install-recommends     build-essential     && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 5000

CMD ["python", "app.py"]
```

### B. Python Facial Feature Model & API (`app.py`)
```python
import os
from flask import Flask, request, jsonify
from model import FaceRecognitionInference

app = Flask(__name__)
engine = FaceRecognitionInference()

@app.route('/mark_attendance', methods=['POST'])
def mark_attendance():
    data = request.get_json(force=True)
    feature_vector = data.get("face_feature_vector")
    result = engine.predict_face(feature_vector)
    return jsonify({"success": True, "record": result})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
```


## 4. Case Study 2: Smart Retail Analytics Platform (Docker Compose Stack)

### `docker-compose.yml` Stack Definition
```yaml
version: '3.8'

services:
  ai-service:
    build: ./ai-service
    container_name: retail_ai_service
    ports:
      - "5000:5000"
    environment:
      - PORT=5000
      - DB_HOST=db
      - DB_NAME=retail_analytics
    depends_on:
      - db
    restart: always

  db:
    image: postgres:15-alpine
    container_name: retail_postgres_db
    environment:
      POSTGRES_DB: retail_analytics
      POSTGRES_USER: postgres
      POSTGRES_PASSWORD: example
    ports:
      - "5432:5432"
    volumes:
      - db_data:/var/lib/postgresql/data

  dashboard:
    build: ./dashboard
    container_name: retail_dashboard_app
    ports:
      - "8501:8501"
    environment:
      - AI_SERVICE_URL=http://ai-service:5000
    depends_on:
      - ai-service
    restart: always

volumes:
  db_data:
```


## 5. GitHub Actions CI/CD Pipeline Workflow

### `.github/workflows/docker-deploy.yml`
```yaml
name: Build and Push Docker Image
on:
  push:
    branches: [ main ]

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Build AI Attendance Image
        run: docker build -t myorg/ai-attendance-system:latest ./ai_attendance_system
      - name: Log in to Docker Hub
        run: echo '${{ secrets.DOCKER_PASSWORD }}' | docker login -u '${{ secrets.DOCKER_USERNAME }}' --password-stdin
      - name: Push image
        run: docker push myorg/ai-attendance-system:latest
```


## 6. Frequently Asked Questions (FAQ Answers)

### Q1: What is containerization, and how does it differ from traditional virtualization?
**Answer:** Containerization is an OS-level virtualization technology that packages an application together with all its runtime dependencies, libraries, binaries, and configurations into an isolated unit called a container. Unlike traditional Virtual Machines (VMs) which require full guest Operating System kernels, containers share the host OS kernel, making them significantly lighter, faster to boot (seconds vs. minutes), and resource-efficient.

### Q2: What is the difference between a Docker image and a Docker container?
**Answer:** 
- **Docker Image**: A read-only, immutable template containing instructions for creating a Docker container (including application code, libraries, and OS environment).
- **Docker Container**: A runnable, isolated instance of a Docker image with a writable container layer. Multiple active containers can run concurrently from a single Docker image.

### Q3: Explain the key instructions used in a Dockerfile: `FROM`, `COPY`, `RUN`, and `CMD`.
**Answer:**
- **`FROM`**: Sets the base image for subsequent instructions (e.g., `FROM python:3.10-slim`).
- **`COPY`**: Copies files/directories from host system into container filesystem (`COPY requirements.txt .`).
- **`RUN`**: Executes build commands inside container during image assembly (`RUN pip install -r requirements.txt`).
- **`CMD`**: Specifies the default executable command when container starts running (`CMD ["python", "app.py"]`).

### Q4: What is Docker Compose, and why is it useful for multi-service AI applications?
**Answer:** Docker Compose is a declarative tool for defining and running multi-container Docker applications using a single YAML file (`docker-compose.yml`). For complex AI systems requiring coordinated components (e.g., an inference API, PostgreSQL database, and Streamlit dashboard), Docker Compose manages container builds, networking, environment variables, startup order (`depends_on`), and volume mounts with a single command (`docker compose up`).

### Q5: How does Docker help achieve scalability in AI system deployment?
**Answer:** Docker containers are lightweight, stateless, and portable, allowing horizontal scaling where multiple instances of an AI service container are spun up behind a load balancer (such as NGINX or Kubernetes ingress) to handle high concurrent traffic spikes seamlessly.

### Q6: What is a CI/CD pipeline, and how does it help automate AI application deployment?
**Answer:** CI/CD (Continuous Integration / Continuous Deployment) automates code checkout, linting, unit testing, Docker image building, and production deployment upon every git push to a repository. Tools like GitHub Actions ensure that broken code or failing AI models are caught before reaching production.

---
## 7. Conclusion
In this practical, we successfully containerized the **AI Attendance Monitoring System** using Docker, orchestrated the **Smart Retail Analytics Platform** using **Docker Compose**, and configured an automated **GitHub Actions CI/CD pipeline**.
